# J5 : End-to-End Image MIL Training on Google Colab

Ce notebook permet d'entraîner un modèle **Multiple Instance Learning (MIL)** directement depuis les images brutes `.jpg`.

**Problème :** Nous n'avons des labels (0 ou 1) qu'au niveau du patient (`bag`), et aucune information sur quelle tuile (1000 tuiles par patient) est cancéreuse ou non. Le MIL résout exactement ce problème.


In [ ]:
# 1. Connection au Drive pour récupérer les images
from google.colab import drive
drive.mount('/content/drive')

# Assurez-vous d'avoir zippé le dossier 'images' original (ex: train_images.zip) 
# et de l'avoir uploadé sur votre Drive.
# !unzip -q /content/drive/MyDrive/OWKIN_data/train_images.zip -d /content/data/
# !cp /content/drive/MyDrive/OWKIN_data/train_output.csv /content/data/


In [ ]:
import os
import glob
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
import random


## 2. Dataset et Dataloader

Pour éviter de saturer la VRAM de Colab avec 1000 images, on utilise une approche d'échantillonnage de `N` tuiles pendant l'entraînement.

In [ ]:
class PatientBagDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, is_train=True, num_samples_per_bag=100):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
        self.is_train = is_train
        self.num_samples_per_bag = num_samples_per_bag
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        patient_id = row['Sample ID'].replace('.npy', '') # ex: ID_001
        label = row['Target']
        
        # Récupérer toutes les tuiles
        patient_folder = os.path.join(self.img_dir, patient_id)
        all_tiles = glob.glob(os.path.join(patient_folder, "*.jpg"))
        
        # Echantillonnage : Charger 100 images aléatoires pour limiter la RAM si en train
        # En validation/test, on pourrait vouloir toutes les 1000, mais ça prend beaucoup de RAM (chunking requis)
        if self.is_train and len(all_tiles) > self.num_samples_per_bag:
            selected_tiles = random.sample(all_tiles, self.num_samples_per_bag)
        else:
            selected_tiles = random.sample(all_tiles, min(len(all_tiles), self.num_samples_per_bag))
            
        images = []
        for img_path in selected_tiles:
            img = Image.open(img_path).convert('RGB')
            if self.transform:
                img = self.transform(img)
            images.append(img)
            
        # Stack les images en un tenseur [num_samples, 3, H, W]
        bag = torch.stack(images)
        label = torch.tensor(label, dtype=torch.float32)
        
        return bag, label

# Data Augmentation standard pour ImageNet
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


## 3. Le Modèle AB-MIL (Attention-Based Multiple Instance Learning)
L'architecture sépare l'extraction de features par un CNN classique et l'agrégation via un mécanisme d'Attention.

In [ ]:
class AttentionMIL(nn.Module):
    def __init__(self, freeze_cnn=True):
        super(AttentionMIL, self).__init__()
        
        # 1. Feature Extractor (Backbone ResNet18 léger)
        weights = models.ResNet18_Weights.IMAGENET1K_V1
        resnet = models.resnet18(weights=weights)
        self.feature_extractor = nn.Sequential(*list(resnet.children())[:-1]) # supprime fc layer
        self.feature_dim = 512
        
        if freeze_cnn:
            for param in self.feature_extractor.parameters():
                param.requires_grad = False
                
        # 2. Attention Mechanism (Ilse et al.)
        self.attention_V = nn.Sequential(
            nn.Linear(self.feature_dim, 128),
            nn.Tanh()
        )
        self.attention_w = nn.Linear(128, 1)
        
        # 3. Classifier final
        self.classifier = nn.Sequential(
            nn.Linear(self.feature_dim, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # x: [bag_size, C, H, W]
        bag_size = x.size(0)
        
        # Extraction de features (CNN)
        h = self.feature_extractor(x)  # [bag_size, 512, 1, 1]
        h = h.view(bag_size, -1)       # [bag_size, 512]
        
        # Calcul de l'Attention (Poids)
        A = self.attention_V(h)  # [bag_size, 128]
        A = self.attention_w(A)  # [bag_size, 1]
        A = torch.transpose(A, 1, 0)  # [1, bag_size]
        A = F.softmax(A, dim=1)  # Softmax over N instances -> [1, bag_size]
        
        # Agrégation (Moyenne pondérée des features selon l'attention)
        M = torch.mm(A, h)  # [1, bag_size] x [bag_size, 512] -> [1, 512]
        
        # Prédiction Finale
        Y_prob = self.classifier(M) # [1, 1]
        
        return Y_prob.squeeze(), A


## 4. Préparation et Boucle d'Entraînement


In [ ]:
# Chemins (à adapter selon là où vous avez unzipé)
IMG_DIR = '/content/data/images'
LABELS_CSV = '/content/data/train_output.csv'

# Création DataFrame fake ou lecture pour qu'il soit exécutable après adaptation
# df = pd.read_csv(LABELS_CSV)
# df_train, df_val = train_test_split(df, test_size=0.2, random_state=42, stratify=df['Target'])

# train_dataset = PatientBagDataset(df_train, IMG_DIR, train_transform, is_train=True, num_samples_per_bag=64)
# val_dataset = PatientBagDataset(df_val, IMG_DIR, val_transform, is_train=False, num_samples_per_bag=128)

# Note: batch_size = 1 (on traite un patient / un bag à la fois)
# train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AttentionMIL(freeze_cnn=True).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)
criterion = nn.BCELoss()

def train(epochs):
    print(f"Training on {device}...")
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        
        # for bag, label in tqdm(train_loader):
        #     bag, label = bag.squeeze(0).to(device), label.to(device) # bag -> [N, 3, 224, 224]
            
        #     optimizer.zero_grad()
        #     prob, A = model(bag)
            
        #     loss = criterion(prob, label[0])
        #     loss.backward()
        #     optimizer.step()
            
        #     train_loss += loss.item()
            
        # val_loss = validate()
        # print(f"Epoch {epoch+1} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss:.4f}")

def validate():
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        pass
        # for bag, label in val_loader:
        #     bag, label = bag.squeeze(0).to(device), label.to(device)
        #     prob, _ = model(bag)
        #     loss = criterion(prob, label[0])
        #     val_loss += loss.item()
    # return val_loss / len(val_loader)
